In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#CONVERTING TO PNG FORMAT

In [ ]:
!pip install pydicom

In [ ]:
import os
import pydicom
import numpy as np
from PIL import Image

# Test Images
def convert_dicom_to_png(input_root):
    """
    Converts all DICOM slices to PNGs and saves them in a 'converted_png'
    folder inside each CT scan folder, alongside 'DICOM_anon' and 'Ground'.
    """
    ct_scan_folders = os.listdir(input_root)

    for ct_scan in ct_scan_folders:
        ct_scan_path = os.path.join(input_root, ct_scan)
        if not os.path.isdir(ct_scan_path):
            continue

        dicom_input_folder = os.path.join(ct_scan_path, 'DICOM_anon')
        png_output_folder = os.path.join(ct_scan_path, 'converted_png')
        os.makedirs(png_output_folder, exist_ok=True)

        if not os.path.isdir(dicom_input_folder):
            print(f"[Skipping] No DICOM_anon in {ct_scan}")
            continue

        dicom_files = [f for f in os.listdir(dicom_input_folder) if f.lower().endswith('.dcm')]

        for dicom_file in dicom_files:
            dicom_path = os.path.join(dicom_input_folder, dicom_file)
            try:
                ds = pydicom.dcmread(dicom_path)
                img_array = ds.pixel_array.astype(float)
                img_array = (np.maximum(img_array, 0) / img_array.max()) * 255.0
                img_array = img_array.astype(np.uint8)
                img = Image.fromarray(img_array)
                slice_num = ds.get('InstanceNumber', dicom_files.index(dicom_file) + 1)
                out_filename = f"slice_{slice_num:03d}.png"
                out_path = os.path.join(png_output_folder, out_filename)
                img.save(out_path)
                print(f"[OK] {dicom_path} -> {out_path}")
            except Exception as e:
                print(f"[Error] {dicom_path}: {e}")

if __name__ == "__main__":
    input_root = "/content/drive/MyDrive/dataset-1/CHAOS_Test_Sets/Test_Sets/CT" # adjust as needed
    convert_dicom_to_png(input_root)


In [ ]:
import os
import pydicom
import numpy as np
from PIL import Image

# Train Images
def convert_dicom_to_png(input_root):
    """
    Converts all DICOM slices to PNGs and saves them in a 'converted_png'
    folder inside each CT scan folder, alongside 'DICOM_anon' and 'Ground'.
    """
    ct_scan_folders = os.listdir(input_root)

    for ct_scan in ct_scan_folders:
        ct_scan_path = os.path.join(input_root, ct_scan)
        if not os.path.isdir(ct_scan_path):
            continue

        dicom_input_folder = os.path.join(ct_scan_path, 'DICOM_anon')
        png_output_folder = os.path.join(ct_scan_path, 'converted_png')
        os.makedirs(png_output_folder, exist_ok=True)

        if not os.path.isdir(dicom_input_folder):
            print(f"[Skipping] No DICOM_anon in {ct_scan}")
            continue

        dicom_files = [f for f in os.listdir(dicom_input_folder) if f.lower().endswith('.dcm')]

        for dicom_file in dicom_files:
            dicom_path = os.path.join(dicom_input_folder, dicom_file)
            try:
                ds = pydicom.dcmread(dicom_path)
                img_array = ds.pixel_array.astype(float)
                img_array = (np.maximum(img_array, 0) / img_array.max()) * 255.0
                img_array = img_array.astype(np.uint8)
                img = Image.fromarray(img_array)
                slice_num = ds.get('InstanceNumber', dicom_files.index(dicom_file) + 1)
                out_filename = f"slice_{slice_num:03d}.png"
                out_path = os.path.join(png_output_folder, out_filename)
                img.save(out_path)
                print(f"[OK] {dicom_path} -> {out_path}")
            except Exception as e:
                print(f"[Error] {dicom_path}: {e}")

if __name__ == "__main__":
    input_root = "/content/drive/MyDrive/dataset-1/CHAOS_Train_Sets/Train_Sets/CT" # adjust as needed
    convert_dicom_to_png(input_root)


In [ ]:
import os
import numpy as np
from PIL import Image

def crop_liver_from_mask(image, mask):
    mask_np = np.array(mask)
    coords = np.argwhere(mask_np)
    if coords.size == 0:
        return None, None
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1
    cropped_img = image.crop((x0, y0, x1, y1))
    cropped_mask = mask.crop((x0, y0, x1, y1))
    return cropped_img, cropped_mask

def preprocess_diffusion_images_and_masks_flat(
    root_folder, output_folder_img, output_folder_mask, target_size=(256,256)
):
    ct_scan_folders = sorted(os.listdir(root_folder))
    os.makedirs(output_folder_img, exist_ok=True)
    os.makedirs(output_folder_mask, exist_ok=True)
    img_counter = 0

    for ct_scan in ct_scan_folders:
        ct_scan_path = os.path.join(root_folder, ct_scan)
        if not os.path.isdir(ct_scan_path):
            continue

        converted_png_folder = os.path.join(ct_scan_path, "converted_png")
        mask_folder = os.path.join(ct_scan_path, "Ground")

        images = sorted([f for f in os.listdir(converted_png_folder) if f.endswith('.png')])
        masks = sorted([f for f in os.listdir(mask_folder) if f.endswith('.png')])

        images = images[:len(masks)]

        for idx, (img_file, mask_file) in enumerate(zip(images, masks)):
            img_path = os.path.join(converted_png_folder, img_file)
            mask_path = os.path.join(mask_folder, mask_file)
            img = Image.open(img_path).convert("L")
            mask = Image.open(mask_path).convert("L")

            cropped_img, cropped_mask = crop_liver_from_mask(img, mask)
            if cropped_img is None:
                print(f"Skipping {img_file}: empty mask")
                continue

            # Resize both image and mask
            resized_img = cropped_img.resize(target_size, Image.BILINEAR)
            resized_mask = cropped_mask.resize(target_size, Image.NEAREST)  # NEAREST for masks to preserve labels

            # Normalize image to [0,1]
            img_np = np.array(resized_img).astype(np.float32) / 255.0
            normalized_img = Image.fromarray((img_np * 255).astype(np.uint8))

            # Save both image and mask
            save_img_path = os.path.join(output_folder_img, f"pre_liver_{img_counter:05d}.png")
            save_mask_path = os.path.join(output_folder_mask, f"pre_mask_{img_counter:05d}.png")
            normalized_img.save(save_img_path)
            resized_mask.save(save_mask_path)
            img_counter += 1
            print(f"Saved: {save_img_path}  |  {save_mask_path}")

if __name__ == "__main__":
    root = "/content/drive/MyDrive/dataset-1/CHAOS_Train_Sets/Train_Sets/CT"
    output_img = "/content/drive/MyDrive/dataset-1/preprocessed_diffusion_input_img"
    output_mask = "/content/drive/MyDrive/dataset-1/preprocessed_diffusion_input_mask"
    preprocess_diffusion_images_and_masks_flat(root, output_img, output_mask, target_size=(256,256))

In [ ]:
import google
from google.colab import drive

# Mount the drive
drive.mount('/content/drive')



In [ ]:
import zipfile
import os

# Extract ZIP file to a folder
zip_path = '/content/drive/MyDrive/hack4health/diffusion_3_preprocessed.zip'
extract_folder = 'extracted_numpy_data'  # Name of folder to create

if zip_path.endswith('.zip') and zipfile.is_zipfile(zip_path):
    print(f"Extracting {zip_path} to {extract_folder}...")

    # Create extraction folder if it doesn't exist
    os.makedirs(extract_folder, exist_ok=True)

    # Extract all contents
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)

    print(f"✅ Extraction complete! Files extracted to: {extract_folder}")

    # Update folder_path to point to extracted folder
    folder_path = extract_folder
else:
    # If not a ZIP file, use as-is
    folder_path = zip_path

# Now use the extracted folder with your dataset

In [ ]:
import os
import shutil

# Root directory where your data is stored
root_dir = "/content/extracted_numpy_data/diffusion_3_preprocessed"
output_dir = "/content/all_images_only"

# Create the output folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Walk through each patient folder
for patient_folder in os.listdir(root_dir):
    patient_path = os.path.join(root_dir, patient_folder)
    images_path = os.path.join(patient_path, "images")

    # Check if "images" folder exists for the patient
    if os.path.isdir(images_path):
        for file in os.listdir(images_path):
            if file.endswith(".npy"):
                src = os.path.join(images_path, file)

                # Add patient name to filename to avoid overwriting
                dst = os.path.join(output_dir, f"{patient_folder}_{file}")
                shutil.copy(src, dst)

print("✅ All .npy files from 'images' folders copied to:", output_dir)


In [ ]:
import shutil

# Path to your Drive (mounted earlier)
drive_path = "/content/drive/MyDrive"

# Source (the new folder you created)
src_folder = "/content/all_images_only"

# Destination (inside your Drive)
dst_folder = os.path.join(drive_path, "all_images_only")

# Copy the folder to Drive
shutil.copytree(src_folder, dst_folder, dirs_exist_ok=True)

print("✅ Folder uploaded to Drive at:", dst_folder)


In [ ]:
import os

folder = "/content/all_images_only"

# Count only files (ignore subfolders, if any)
num_files = len([f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))])

print("Number of files in folder:", num_files)


In [ ]:
# @title DIFFUSION MODEL ,DDPM
#diffusion model
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Updated Dataset class for multiple numpy files in a folder
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True):
        """
        Args:
            folder_path: Path to folder containing .npy files
            augment: Whether to apply data augmentation
        """
        print(f"Loading numpy files from folder: {folder_path}")

        # Find all .npy files in the folder
        npy_files = glob.glob(os.path.join(folder_path, "*.npy"))

        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")

        # Load and concatenate all numpy arrays
        all_images = []
        for npy_file in tqdm(npy_files, desc="Loading .npy files"):
            images = np.load(npy_file)
            print(f"Loaded {npy_file}: shape {images.shape}")

            # Handle different shapes
            if len(images.shape) == 2:  # Single image (H, W)
                images = images[None, :, :]  # Add batch dimension -> (1, H, W)
            elif len(images.shape) == 3:  # Could be (N, H, W) or (H, W, C)
                if images.shape[-1] == 1 or images.shape[-1] == 3:  # (H, W, C)
                    images = images[None, :, :, :]  # Add batch dimension -> (1, H, W, C)

            all_images.append(images)

        # Concatenate all images
        self.images = np.concatenate(all_images, axis=0)

        print(f"Combined dataset: {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")

        # Normalize images to [0, 1] if not already normalized
        if self.images.max() > 1.0:
            print("Normalizing images from [0, 255] to [0, 1]")
            self.images = self.images.astype(np.float32) / 255.0
        else:
            self.images = self.images.astype(np.float32)

        # Ensure images have the right dimensions for grayscale
        if len(self.images.shape) == 3:  # (N, H, W)
            print("Adding channel dimension for grayscale")
            self.images = self.images[:, None, :, :]  # (N, 1, H, W)
        elif len(self.images.shape) == 4:
            if self.images.shape[-1] == 1:  # (N, H, W, 1) -> (N, 1, H, W)
                print("Converting from (N, H, W, 1) to (N, 1, H, W)")
                self.images = np.transpose(self.images, (0, 3, 1, 2))
            elif self.images.shape[1] != 1 and self.images.shape[1] != 3:  # (N, H, W, C) format
                print("Converting from (N, H, W, C) to (N, C, H, W)")
                self.images = np.transpose(self.images, (0, 3, 1, 2))

        print(f"Final image shape: {self.images.shape}")

        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()  # (1, H, W) or (C, H, W)

        # Ensure image is (1, H, W) for grayscale
        if len(image.shape) == 2:
            image = image[None, :, :]
        elif len(image.shape) == 3 and image.shape[0] != 1:
            # If it's RGB, convert to grayscale
            if image.shape[0] == 3:
                image = np.mean(image, axis=0, keepdims=True)

        # Convert to tensor
        image = torch.from_numpy(image).float()

        # Apply augmentation
        if self.augment and np.random.random() > 0.5:
            # Random horizontal flip
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

            # Random vertical flip (careful with CT scans - may not be appropriate)
            # if np.random.random() > 0.7:
            #     image = torch.flip(image, [1])

        # Normalize to [-1, 1] for diffusion training
        image = image * 2.0 - 1.0

        return image

# Alternative: Single file dataset (your original code)
class SingleNumpyDataset(Dataset):
    def __init__(self, numpy_file_path, augment=True):
        """
        For when you have a single .npy file containing all images
        Args:
            numpy_file_path: Path to single .npy file containing images
            augment: Whether to apply data augmentation
        """
        print(f"Loading numpy array from single file: {numpy_file_path}")

        # Load numpy array
        self.images = np.load(numpy_file_path)

        print(f"Loaded {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")

        # Normalize images to [0, 1] if not already normalized
        if self.images.max() > 1.0:
            print("Normalizing images from [0, 255] to [0, 1]")
            self.images = self.images.astype(np.float32) / 255.0
        else:
            self.images = self.images.astype(np.float32)

        # Ensure images have the right dimensions
        if len(self.images.shape) == 3:  # (N, H, W)
            print("Adding channel dimension")
            self.images = self.images[:, None, :, :]  # (N, 1, H, W)
        elif len(self.images.shape) == 4 and self.images.shape[-1] == 1:  # (N, H, W, 1)
            print("Converting from (N, H, W, 1) to (N, 1, H, W)")
            self.images = np.transpose(self.images, (0, 3, 1, 2))

        print(f"Final image shape: {self.images.shape}")

        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()  # (1, H, W) or (H, W)

        # Ensure image is (1, H, W)
        if len(image.shape) == 2:
            image = image[None, :, :]

        # Convert to tensor
        image = torch.from_numpy(image).float()

        # Apply augmentation
        if self.augment and np.random.random() > 0.5:
            # Random horizontal flip
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        # Normalize to [-1, 1] for diffusion training
        image = image * 2.0 - 1.0

        return image

# Choose the appropriate dataset based on your setup
folder_path = '/content/all_images_only'  # Folder containing multiple .npy files

# OPTION 1: If you have multiple .npy files in a folder (your case)
train_dataset = MultipleNumpyDataset(folder_path, augment=True)

# OPTION 2: If you have a single .npy file containing all images
# single_file_path = 'path/to/your/single/file.npy'
# train_dataset = SingleNumpyDataset(single_file_path, augment=True)

# Rest of your training code remains the same...
train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(train_dataloader)}")

# The rest of your training code continues unchanged...



In [ ]:


# Model configuration optimized for 256x256 CT scans
model = UNet2DModel(
    sample_size=256,      # Your image size
    in_channels=1,        # Grayscale CT scans
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512),  # Full capacity for better quality
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)

# Print model information
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.1f} MB")

# Scheduler optimized for medical images
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="scaled_linear",
    prediction_type="epsilon",
    clip_sample=False  # Don't clip for better quality
)

# Optimizer with medical image-specific settings
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01,
    eps=1e-8,
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

# Mixed precision for memory efficiency
scaler = GradScaler()

# Training parameters
num_epochs = 10  # More epochs for better convergence
accumulation_steps = 8  # Effective batch size: 4 * 8 = 32
save_every = 2
log_every = 50

# Check GPU memory before training
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB allocated")
    print(f"GPU Memory: {torch.cuda.memory_reserved()/1024**3:.2f}GB reserved")

# Training loop
global_step = 0
best_loss = float('inf')
train_losses = []

print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    progress_bar = tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)

        # Sample random timesteps
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean_images.shape[0],),
            device=device
        ).long()

        # Add noise to the clean images
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        # Forward pass with mixed precision
        with autocast():
            # Predict the noise residual
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]

            # Calculate loss (MSE between predicted and actual noise)
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        # Backward pass
        scaler.scale(loss).backward()

        # Update parameters after accumulation steps
        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            # Gradient clipping for stability
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        # Track metrics
        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1

        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })

        global_step += 1

        # Log and clear memory periodically
        if batch_idx % log_every == 0:
            train_losses.append(batch_loss)
            clear_memory()

    # End of epoch processing
    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    # Save best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss,
            'global_step': global_step
        }
        torch.save(checkpoint, "ddpm_ct_best_model.pt")
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    # Save periodic checkpoints
    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss,
            'global_step': global_step
        }
        torch.save(checkpoint, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt")
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    # GPU memory check
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

# Final model save
torch.save(model.state_dict(), "ddpm_ct_final_model.pt")
print("🎉 Training completed!")

# Plot training loss
if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

# Test the model with sample generation
print("🖼️  Generating test sample...")
model.eval()
with torch.no_grad():
    # Generate a sample
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)

    # Denoising process
    timesteps = noise_scheduler.timesteps
    for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample

        # Show progress every 100 steps
        if i % 100 == 0:
            clear_memory()

    # Convert to numpy and save
    sample = (sample + 1) / 2  # Convert from [-1, 1] to [0, 1]
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]  # Remove batch and channel dims

    # Save sample
    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")

# Instructions for resuming training
print("\n🔄 To resume training from a checkpoint:")
print("""
checkpoint = torch.load('ddmp_ct_best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1
""")

clear_memory()
print("🏁 All done!")

In [ ]:
#modified diffusion model lightweight

import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Data Loading Classes (Your code, unchanged) ---
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True):
        print(f"Loading numpy files from folder: {folder_path}")
        npy_files = glob.glob(os.path.join(folder_path, "*.npy"))
        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found in {folder_path}")
        print(f"Found {len(npy_files)} .npy files")
        all_images = []
        for npy_file in tqdm(npy_files, desc="Loading .npy files"):
            images = np.load(npy_file)
            print(f"Loaded {npy_file}: shape {images.shape}")
            if len(images.shape) == 2:
                images = images[None, :, :]
            elif len(images.shape) == 3:
                if images.shape[-1] == 1 or images.shape[-1] == 3:
                    images = images[None, :, :, :]
            all_images.append(images)
        self.images = np.concatenate(all_images, axis=0)
        print(f"Combined dataset: {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")
        if self.images.max() > 1.0:
            print("Normalizing images from [0, 255] to [0, 1]")
            self.images = self.images.astype(np.float32) / 255.0
        else:
            self.images = self.images.astype(np.float32)
        if len(self.images.shape) == 3:
            print("Adding channel dimension for grayscale")
            self.images = self.images[:, None, :, :]
        elif len(self.images.shape) == 4:
            if self.images.shape[-1] == 1:
                print("Converting from (N, H, W, 1) to (N, 1, H, W)")
                self.images = np.transpose(self.images, (0, 3, 1, 2))
            elif self.images.shape[1] != 1 and self.images.shape[1] != 3:
                print("Converting from (N, H, W, C) to (N, C, H, W)")
                self.images = np.transpose(self.images, (0, 3, 1, 2))
        print(f"Final image shape: {self.images.shape}")
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()
        if len(image.shape) == 2:
            image = image[None, :, :]
        elif len(image.shape) == 3 and image.shape[0] != 1:
            if image.shape[0] == 3:
                image = np.mean(image, axis=0, keepdims=True)
        image = torch.from_numpy(image).float()
        if self.augment and np.random.random() > 0.5:
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])
        image = image * 2.0 - 1.0
        return image

# --- Dataset and DataLoader Setup ---
folder_path = 'path/to/your/numpy/folder'
train_dataset = MultipleNumpyDataset(folder_path, augment=True)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8, # Your mini-batch size
    shuffle=True,
    num_workers=4, # INCREASED from 2 for faster data loading
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)
print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(train_dataloader)}")

# --- Model Configuration ---
model = UNet2DModel(
    sample_size=256,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 128, 256), # REDUCED for faster training
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.1f} MB")

# --- Training Setup ---
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="scaled_linear",
    prediction_type="epsilon",
    clip_sample=False
)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01,
    eps=1e-8,
    betas=(0.9, 0.999)
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler()

# Training parameters
num_epochs = 10
accumulation_steps = 4 # Your effective batch size: 8 * 4 = 32
save_every = 2
log_every = 50

# --- Training Loop ---
global_step = 0
best_loss = float('inf')
train_losses = []
print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")
    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)
        with autocast():
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps
        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1
        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })
        global_step += 1
        if batch_idx % log_every == 0:
            train_losses.append(batch_loss)
            clear_memory()

    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()
    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss,
            'global_step': global_step
        }
        torch.save(checkpoint, "ddpm_ct_best_model.pt")
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss,
            'global_step': global_step
        }
        torch.save(checkpoint, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt")
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

torch.save(model.state_dict(), "ddpm_ct_final_model.pt")
print("🎉 Training completed!")

if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

print("🖼️  Generating test sample...")
model.eval()
with torch.no_grad():
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)
    timesteps = noise_scheduler.timesteps
    for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample
        if i % 100 == 0:
            clear_memory()
    sample = (sample + 1) / 2
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]
    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")
print("\n🔄 To resume training from a checkpoint:")
print("""
checkpoint = torch.load('ddpm_ct_best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1
""")
clear_memory()
print("🏁 All done!")



In [ ]:
#diffusion model optimized code

import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Updated Dataset class for multiple numpy files in a folder
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True, target_shape=(1, 256, 256)):
        """
        Args:
            folder_path: Path to folder containing .npy files (can be nested)
            augment: Whether to apply data augmentation
            target_shape: The desired shape for all images (C, H, W)
        """
        print(f"Loading numpy files from folder: {folder_path}")
        self.target_shape = target_shape

        # Find all .npy files recursively in the folder
        npy_files = glob.iglob(os.path.join(folder_path, "**", "*.npy"), recursive=True)
        npy_files = list(npy_files) # Convert generator to list to check count

        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found recursively in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")

        # Load and process numpy arrays directly into memory
        all_images_list = [] # Use a list to collect images before concatenating
        skipped_files = []

        for npy_file in tqdm(npy_files, desc="Loading and processing .npy files"):
            try:
                images = np.load(npy_file)
                original_shape = images.shape
                # print(f"Loaded {npy_file}: shape {original_shape}") # Commented out to reduce verbose output

                # --- Shape Handling and Filtering ---
                # Ensure images have the right dimensions for grayscale (N, 1, H, W)
                if len(images.shape) == 2:  # Single image (H, W)
                    images = images[None, None, :, :]  # Add batch and channel dimension -> (1, 1, H, W)
                elif len(images.shape) == 3:  # Could be (N, H, W) or (H, W, C)
                    if images.shape[-1] == 1: # (H, W, 1)
                         images = images[None, :, :, :] # Add batch dim -> (1, H, W, 1)
                         images = np.transpose(images, (0, 3, 1, 2)) # -> (1, 1, H, W)
                    elif images.shape[-1] == 3: # (H, W, 3)
                         images = images[None, :, :, :] # Add batch dim -> (1, H, W, 3)
                         images = np.mean(images, axis=-1, keepdims=True) # -> (1, H, W, 1)
                         images = np.transpose(images, (0, 3, 1, 2)) # -> (1, 1, H, W)
                    elif images.shape[0] > 1: # (N, H, W)
                         images = images[:, None, :, :] # Add channel dimension -> (N, 1, H, W)
                    else:
                         print(f"Skipping {npy_file}: Unexpected 3D shape {original_shape}")
                         skipped_files.append(npy_file)
                         continue
                elif len(images.shape) == 4: # Assuming (N, H, W, C) or (N, C, H, W)
                    if images.shape[1] > 1: # Assuming (N, C, H, W) or (N, H, W, C)
                         if images.shape[1] == 3: # If RGB (N, 3, H, W), convert to grayscale
                             images = np.mean(images, axis=1, keepdims=True) # (N, 1, H, W)
                         elif images.shape[-1] == 3: # If RGB (N, H, W, 3), convert to grayscale
                              images = np.mean(images, axis=-1, keepdims=True) # (N, H, W, 1)
                              images = np.transpose(images, (0, 3, 1, 2)) # (N, 1, H, W)
                         elif images.shape[-1] == 1: # If (N, H, W, 1), convert to (N, 1, H, W)
                              images = np.transpose(images, (0, 3, 1, 2))
                         elif images.shape[1] == 1: # Already (N, 1, H, W)
                             pass
                         else:
                             print(f"Skipping {npy_file}: Unexpected 4D shape with more than 1 channel {images.shape}")
                             skipped_files.append(npy_file)
                             continue
                    elif images.shape[1] == 1: # Already (N, 1, H, W)
                        pass
                else:
                     print(f"Skipping {npy_file}: Unexpected shape {original_shape}")
                     skipped_files.append(npy_file)
                     continue # Skip files with unexpected dimensions

                # --- Resize/Pad to Target Shape if necessary ---
                # This part is crucial for preloading if image sizes vary
                # The original code assumed all images were already target size.
                # To handle varying sizes, we would need resizing/padding logic here.
                # For this subtask, assuming images are already the target size (256x256)
                # based on the previous cell's output showing (2341, 1, 256, 256).
                # If sizes varied, this would need significant modification.

                # Check if the processed image batch matches the target shape (C, H, W) for each image in the batch
                if len(images.shape) == 4 and images.shape[1:] == self.target_shape:
                    # Normalize images to [0, 1] if not already normalized
                    if images.max() > 1.0:
                        images = images.astype(np.float32) / 255.0
                    else:
                         images = images.astype(np.float32) # Ensure float32 even if already normalized

                    all_images_list.append(images) # Append the batch of images
                else:
                     print(f"Skipping {npy_file}: Final shape {images.shape} does not match target batch shape (N, {self.target_shape})")
                     skipped_files.append(npy_file)


            except Exception as e:
                print(f"Error loading or processing {npy_file}: {e}")
                skipped_files.append(npy_file)
                continue # Skip files that cause errors

        if len(all_images_list) == 0:
             raise ValueError(f"No valid images loaded from {folder_path}. {len(skipped_files)} files were skipped.")


        print(f"\nSuccessfully loaded and processed {len(all_images_list)} image batches.")
        print(f"Skipped {len(skipped_files)} files due to errors or incorrect shape.")

        # Concatenate all images into a single numpy array
        try:
            self.images = np.concatenate(all_images_list, axis=0)
        except ValueError as e:
             print(f"Error concatenating images: {e}")
             # This should ideally not happen now if filtering worked correctly
             shapes = [img.shape for img in all_images_list]
             print(f"Encountered images with inconsistent shapes during final concat: {shapes}")
             raise ValueError("Images still have inconsistent shapes and cannot be concatenated after filtering.") from e


        print(f"Combined dataset (preloaded): {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")

        # The images are already normalized to [0, 1] and are float32 from the loading loop
        # No need for the separate normalization/dtype check here unless debugging

        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Access the preloaded image directly
        image = self.images[idx].copy()  # (1, H, W) - copy to avoid modifying the original array

        # Ensure image is (1, H, W) for grayscale before returning (should already be this from init)
        # This check should ideally not be needed if __init__ is correct, but keep for safety
        if len(image.shape) == 3 and image.shape[0] != 1:
            if image.shape[0] == 3:
                image = np.mean(image, axis=0, keepdims=True)
            else:
                 print(f"Warning: Image in getitem has unexpected channel count ({image.shape[0]}). Expected 1.")
                 # Handle unexpected shape - e.g., return a zero tensor or raise error
                 # For now, let's just print a warning and hope it's not frequent

        # Convert to tensor
        image = torch.from_numpy(image).float() # Should already be float32 from init, but .float() is safe

        # Apply augmentation
        if self.augment and np.random.random() > 0.5:
            # Random horizontal flip
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        # Normalize to [-1, 1] for diffusion training
        image = image * 2.0 - 1.0

        return image

# Alternative: Single file dataset (your original code) - Keep for reference, but not used in this subtask
class SingleNumpyDataset(Dataset):
    def __init__(self, numpy_file_path, augment=True):
        """
        For when you have a single .npy file containing all images
        Args:
            numpy_file_path: Path to single .npy file containing images
            augment: Whether to apply data augmentation
        """
        print(f"Loading numpy array from single file: {numpy_file_path}")

        # Load numpy array
        self.images = np.load(numpy_file_path)

        print(f"Loaded {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")

        # Normalize images to [0, 1] if not already normalized
        if self.images.max() > 1.0:
            print("Normalizing images from [0, 255] to [0, 1]")
            self.images = self.images.astype(np.float32) / 255.0
        else:
            self.images = self.images.astype(np.float32)

        # Ensure images have the right dimensions
        if len(self.images.shape) == 3:  # (N, H, W)
            print("Adding channel dimension")
            self.images = self.images[:, None, :, :]  # (N, 1, H, W)
        elif len(self.images.shape) == 4 and self.images.shape[-1] == 1:  # (N, H, W, 1)
            print("Converting from (N, H, W, 1) to (N, 1, H, W)")
            self.images = np.transpose(self.images, (0, 3, 1, 2))

        print(f"Final image shape: {self.images.shape}")

        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()  # (1, H, W) or (H, W)

        # Ensure image is (1, H, W)
        if len(image.shape) == 2:
            image = image[None, :, :]

        # Convert to tensor
        image = torch.from_numpy(image).float()

        # Apply augmentation
        if self.augment and np.random.random() > 0.5:
            # Random horizontal flip
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        # Normalize to [-1, 1] for diffusion training
        image = image * 2.0 - 1.0

        return image


# Choose the appropriate dataset based on your setup
# Update this path to point to the directory containing the nested image folders
folder_path = "/content/drive/MyDrive/all_images_only" # UPDATED PATH

# OPTION 1: If you have multiple .npy files in a folder (your case)
train_dataset = MultipleNumpyDataset(folder_path, augment=True, target_shape=(1, 256, 256)) # Added target_shape

# OPTION 2: If you have a single .npy file containing all images
# single_file_path = 'path/to/your/single/file.npy'
# train_dataset = SingleNumpyDataset(single_file_path, augment=True)

# Rest of your training code remains the same...
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8, # Increased batch size
    shuffle=True,
    num_workers=4, # Increased num_workers
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

print(f"Dataset size: {len(train_dataset)}")
print(f"Number of batches per epoch: {len(train_dataloader)}")

# The rest of your training code continues unchanged...

# Model configuration optimized for 256x256 CT scans (slightly reduced for free tier)
model = UNet2DModel(
    sample_size=256,        # Your image size - Consider reducing this (e.g., to 128) for significant memory savings on free tier
    in_channels=1,          # Grayscale CT scans
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),  # Slightly reduced capacity
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D", "DownBlock2D"), # Reduced attention blocks
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"), # Reduced attention blocks
    attention_head_dim=8,
)
model.to(device)

# Print model information
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.1f} MB")


# Scheduler optimized for medical images
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="scaled_linear",
    prediction_type="epsilon",
    clip_sample=False  # Don't clip for better quality
)

# Optimizer with medical image-specific settings
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4, # May need slight adjustment with reduced batch size/accumulation
    weight_decay=0.01,
    eps=1e-8,
    betas=(0.9, 0.999)
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10) # Adjust T_max based on num_epochs

# Mixed precision for memory efficiency
scaler = GradScaler()

# Training parameters
num_epochs = 15  # Increased epochs to compensate for smaller batch size per step
accumulation_steps = 4  # Reduced accumulation steps (8 * 4 = 32 effective batch size)
save_every = 2
log_every = 20 # Increased log frequency for better monitoring

# Check GPU memory before training
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB allocated")
    print(f"GPU Memory: {torch.cuda.memory_reserved()/1024**3:.2f}GB reserved")

# Training loop
global_step = 0
best_loss = float('inf')
train_losses = []

print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0

    progress_bar = tqdm(
        enumerate(train_dataloader),
        total=len(train_dataloader),
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)

        # Sample random timesteps
        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (clean_images.shape[0],),
            device=device
        ).long()

        # Add noise to the clean images
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        # Forward pass with mixed precision
        with autocast():
            # Predict the noise residual
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]

            # Calculate loss (MSE between predicted and actual noise)
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        # Backward pass
        scaler.scale(loss).backward()

        # Update parameters after accumulation steps
        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            # Gradient clipping for stability
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            # Clear memory after parameter update
            clear_memory()


        # Track metrics
        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1

        # Update progress bar
        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })

        global_step += 1

        # Log periodically
        if batch_idx % log_every == 0:
             train_losses.append(batch_loss)
             # clear_memory() # Removed this clear_memory as it's now after scaler.step

    # End of epoch processing
    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    # Save best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss,
            'global_step': global_step
        }
        torch.save(checkpoint, "ddpm_ct_best_model.pt")
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    # Save periodic checkpoints
    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss,
            'global_step': global_step
        }
        torch.save(checkpoint, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt")
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    # GPU memory check
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

# Final model save
torch.save(model.state_dict(), "ddpm_ct_final_model.pt")
print("🎉 Training completed!")

# Plot training loss
if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

# Test the model with sample generation
'''print("🖼️  Generating test sample...")
model.eval()
with torch.no_grad():
    # Generate a sample
    sample_shape = (1, 1, 256, 256) # Note: If you reduced training image size, change this too
    sample = torch.randn(sample_shape, device=device)

    # Denoising process
    timesteps = noise_scheduler.timesteps
    for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample

        # Show progress every 100 steps
        if i % 100 == 0: # Increased clear_memory frequency during inference
            clear_memory()


    # Convert to numpy and save
    sample = (sample + 1) / 2  # Convert from [-1, 1] to [0, 1]
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]  # Remove batch and channel dims

    # Save sample
    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256") # Update if you reduce resolution
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")

# Instructions for resuming training
print("\n🔄 To resume training from a checkpoint:")
print("""
checkpoint = torch.load('ddmp_ct_best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1
# You might also need to load the scaler state if using mixed precision
# scaler.load_state_dict(checkpoint['scaler_state_dict'])
""")


clear_memory()
print("🏁 All done!")'''

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Updated Dataset class for multiple numpy files in a folder
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True, target_shape=(1, 256, 256)):
        print(f"Loading numpy files from folder: {folder_path}")
        self.target_shape = target_shape
        npy_files = list(glob.iglob(os.path.join(folder_path, "**", "*.npy"), recursive=True))

        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found recursively in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")
        all_images_list = []
        skipped_files = []

        for npy_file in tqdm(npy_files, desc="Loading and processing .npy files"):
            try:
                images = np.load(npy_file)
                original_shape = images.shape
                if len(images.shape) == 2:
                    images = images[None, None, :, :]
                elif len(images.shape) == 3:
                    if images.shape[-1] == 1:
                        images = images[None, :, :, :]
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[-1] == 3:
                        images = images[None, :, :, :]
                        images = np.mean(images, axis=-1, keepdims=True)
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[0] > 1:
                        images = images[:, None, :, :]
                    else:
                        print(f"Skipping {npy_file}: Unexpected 3D shape {original_shape}")
                        skipped_files.append(npy_file)
                        continue
                elif len(images.shape) == 4:
                    if images.shape[1] > 1:
                        if images.shape[1] == 3:
                            images = np.mean(images, axis=1, keepdims=True)
                        elif images.shape[-1] == 3:
                            images = np.mean(images, axis=-1, keepdims=True)
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[-1] == 1:
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[1] == 1:
                            pass
                        else:
                            print(f"Skipping {npy_file}: Unexpected 4D shape with more than 1 channel {images.shape}")
                            skipped_files.append(npy_file)
                            continue
                    elif images.shape[1] == 1:
                        pass
                else:
                    print(f"Skipping {npy_file}: Unexpected shape {original_shape}")
                    skipped_files.append(npy_file)
                    continue

                if len(images.shape) == 4 and images.shape[1:] == self.target_shape:
                    if images.max() > 1.0:
                        images = images.astype(np.float32) / 255.0
                    else:
                        images = images.astype(np.float32)
                    all_images_list.append(images)
                else:
                    print(f"Skipping {npy_file}: Final shape {images.shape} does not match target batch shape (N, {self.target_shape})")
                    skipped_files.append(npy_file)

            except Exception as e:
                print(f"Error loading or processing {npy_file}: {e}")
                skipped_files.append(npy_file)
                continue

        if len(all_images_list) == 0:
            raise ValueError(f"No valid images loaded from {folder_path}. {len(skipped_files)} files were skipped.")

        print(f"\nSuccessfully loaded and processed {len(all_images_list)} image batches.")
        print(f"Skipped {len(skipped_files)} files due to errors or incorrect shape.")

        try:
            self.images = np.concatenate(all_images_list, axis=0)
        except ValueError as e:
            print(f"Error concatenating images: {e}")
            shapes = [img.shape for img in all_images_list]
            print(f"Encountered images with inconsistent shapes during final concat: {shapes}")
            raise ValueError("Images still have inconsistent shapes and cannot be concatenated after filtering.") from e

        print(f"Combined dataset (preloaded): {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()
        image = torch.from_numpy(image).float()

        if self.augment and np.random.random() > 0.5:
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        return image * 2.0 - 1.0

# --- Main Script ---
# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Dataset and DataLoader setup
folder_path = '/content/drive/MyDrive/dataset-1/all_images_only'
train_dataset = MultipleNumpyDataset(folder_path, augment=True)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

# Model configuration
model = UNet2DModel(
    sample_size=256,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(32, 64, 128, 256), # REDUCED for faster training
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)

# Scheduler, Optimizer, Scaler, and Checkpointing
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler()
best_loss = float('inf')

# --- Checkpoint Resuming Section (Modified for your specific case) ---
checkpoint_path = "/content/drive/MyDrive/dataset-1/ddpm_ct_checkpoint_epoch_2.pt" # Your specified checkpoint file
print(f"Loading checkpoint from {checkpoint_path}...")
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_epoch = checkpoint['epoch'] + 1 # This will be 2 + 1 = 3
best_loss = checkpoint['loss']
print(f"Resuming training from epoch {start_epoch}")

# Training parameters (Modified)
num_epochs = start_epoch + 3 # Train for 3 more epochs, so num_epochs will be 6
accumulation_steps = 4
save_every = 2
log_every = 20
global_step = checkpoint['global_step'] if 'global_step' in checkpoint else 0

# --- Training Loop ---
print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        with autocast():
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            clear_memory()

        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1

        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })
        global_step += 1

        if batch_idx % log_every == 0:
            pass # Removed logging the loss per batch here

    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss, 'global_step': global_step
        }
        torch.save(checkpoint, "ddpm_ct_best_model.pt")
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss, 'global_step': global_step
        }
        torch.save(checkpoint, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt")
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

torch.save(model.state_dict(), "ddpm_ct_final_model.pt")
print("🎉 Training completed!")

# --- Inference Section ---
print("🖼  Generating test sample...")
model.eval()
noise_scheduler.set_timesteps(num_inference_steps=50) # Use the DDPMScheduler for inference

with torch.no_grad():
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)

    for i, t in enumerate(tqdm(noise_scheduler.timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample
        if i % 10 == 0: clear_memory()

    sample = (sample + 1) / 2
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]

    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")
print("\n🏁 All done!")

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Updated Dataset class for multiple numpy files in a folder
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True, target_shape=(1, 256, 256)):
        print(f"Loading numpy files from folder: {folder_path}")
        self.target_shape = target_shape
        npy_files = list(glob.iglob(os.path.join(folder_path, "", "*.npy"), recursive=True))

        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found recursively in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")
        all_images_list = []
        skipped_files = []

        for npy_file in tqdm(npy_files, desc="Loading and processing .npy files"):
            try:
                images = np.load(npy_file)
                original_shape = images.shape
                if len(images.shape) == 2:
                    images = images[None, None, :, :]
                elif len(images.shape) == 3:
                    if images.shape[-1] == 1:
                        images = images[None, :, :, :]
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[-1] == 3:
                        images = images[None, :, :, :]
                        images = np.mean(images, axis=-1, keepdims=True)
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[0] > 1:
                        images = images[:, None, :, :]
                    else:
                        print(f"Skipping {npy_file}: Unexpected 3D shape {original_shape}")
                        skipped_files.append(npy_file)
                        continue
                elif len(images.shape) == 4:
                    if images.shape[1] > 1:
                        if images.shape[1] == 3:
                            images = np.mean(images, axis=1, keepdims=True)
                        elif images.shape[-1] == 3:
                            images = np.mean(images, axis=-1, keepdims=True)
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[-1] == 1:
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[1] == 1:
                            pass
                        else:
                            print(f"Skipping {npy_file}: Unexpected 4D shape with more than 1 channel {images.shape}")
                            skipped_files.append(npy_file)
                            continue
                    elif images.shape[1] == 1:
                        pass
                else:
                    print(f"Skipping {npy_file}: Unexpected shape {original_shape}")
                    skipped_files.append(npy_file)
                    continue

                if len(images.shape) == 4 and images.shape[1:] == self.target_shape:
                    if images.max() > 1.0:
                        images = images.astype(np.float32) / 255.0
                    else:
                        images = images.astype(np.float32)
                    all_images_list.append(images)
                else:
                    print(f"Skipping {npy_file}: Final shape {images.shape} does not match target batch shape (N, {self.target_shape})")
                    skipped_files.append(npy_file)

            except Exception as e:
                print(f"Error loading or processing {npy_file}: {e}")
                skipped_files.append(npy_file)
                continue

        if len(all_images_list) == 0:
            raise ValueError(f"No valid images loaded from {folder_path}. {len(skipped_files)} files were skipped.")

        try:
            self.images = np.concatenate(all_images_list, axis=0)
        except ValueError as e:
            print(f"Error concatenating images: {e}")
            shapes = [img.shape for img in all_images_list]
            print(f"Encountered images with inconsistent shapes during final concat: {shapes}")
            raise ValueError("Images still have inconsistent shapes and cannot be concatenated after filtering.") from e

        print(f"Combined dataset (preloaded): {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()
        image = torch.from_numpy(image).float()

        if self.augment and np.random.random() > 0.5:
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        return image * 2.0 - 1.0

# --- Main Script ---
# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Dataset and DataLoader setup
folder_path = '/content/drive/MyDrive/dataset-1/all_images_only'
train_dataset = MultipleNumpyDataset(folder_path, augment=True)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

# Model configuration
model = UNet2DModel(
    sample_size=256,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)

# Scheduler, Optimizer, Scaler, and Checkpointing
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler()
best_loss = float('inf')

# --- Checkpoint Resuming Section ---
checkpoint_path = "ddpm_ct_checkpoint_epoch_2.pt"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['loss']
    print(f"Resuming training from epoch {start_epoch}")
else:
    print("No checkpoint found. Starting training from scratch.")
    start_epoch = 0

# Training parameters
num_epochs = start_epoch + 3
accumulation_steps = 4
save_every = 2
log_every = 20
global_step = checkpoint['global_step'] if os.path.exists(checkpoint_path) and 'global_step' in checkpoint else 0

# --- Training Loop ---
print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        with autocast():
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            clear_memory()

        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1

        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })
        global_step += 1

        if batch_idx % log_every == 0:
            pass

    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss, 'global_step': global_step
        }
        torch.save(checkpoint, "ddpm_ct_best_model.pt")
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss, 'global_step': global_step
        }
        torch.save(checkpoint, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt")
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

torch.save(model.state_dict(), "ddpm_ct_final_model.pt")
print("🎉 Training completed!")

# Plot training loss
if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

# Test the model with sample generation
print("🖼  Generating test sample...")
model.eval()
with torch.no_grad():
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)
    timesteps = noise_scheduler.timesteps
    for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample
        if i % 100 == 0:
            clear_memory()

    sample = (sample + 1) / 2
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]

    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")
print("\n🏁 All done!")

CNN TESTING

In [ ]:
# --- PLEASE UPDATE THESE PATHS ---

# Path to your trained .keras model file
MODEL_PATH = '/content/drive/MyDrive/dataset-1/liver_unet.keras'

# Path to the parent folder containing your patient folders (e.g., Patient_13, etc.)
DATA_DIR = '/content/drive/MyDrive/dataset-1/Preprocessed_CNN_TEST'

# Directory where the prediction images will be saved
RESULTS_DIR = '/content/drive/MyDrive/dataset-1/Result CNN Test'

In [ ]:
# --- Cell 2: The Main Prediction Script (Tests All NPZ Files with Patient ID Naming) ---
import os
import sys
import glob
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import backend as K

# --- CUSTOM FUNCTIONS (Needed to load the model) ---
def dice_coef(y_true, y_pred, smooth=1): return None
def dice_loss(y_true, y_pred): return None
def combined_loss(y_true, y_pred): return None

# --- MAIN WORKFLOW ---
def main(model_path, data_dir, results_dir):
    print("🚀 Starting U-Net Prediction on ALL NPZ Slices 🚀"); print("=" * 60)

    # 1. Load the U-Net Model
    print(f"Loading model from: {model_path}")
    try:
        model = keras.models.load_model(
            model_path,
            custom_objects={'dice_coef': dice_coef, 'dice_loss': dice_loss, 'combined_loss': combined_loss},
            safe_mode=False,
            compile=False
        )
        print("✅ Model loaded successfully!")
        target_size = model.input_shape[1:3]
        print(f"Model expects input size: {target_size}")
    except Exception as e:
        print(f"❌ Error loading model: {e}"); return

    # 2. Find all NPZ files
    npz_files = list(glob.glob(os.path.join(data_dir, '**', '*.npz'), recursive=True))
    if not npz_files:
        print(f"❌ Error: No .npz files found in {data_dir}."); return
    total_files = len(npz_files)
    print(f"Found {total_files} total .npz slice files to process.")

    os.makedirs(results_dir, exist_ok=True)

    # 3. Process every file
    for i, file_path in enumerate(npz_files):
        print(f"--- Processing file {i+1}/{total_files}: {os.path.basename(file_path)} ---")

        # Load the NPZ file
        try:
            with np.load(file_path) as data:
                if 'image' in data: ct_slice = data['image']
                elif 'data' in data: ct_slice = data['data']
                elif 'arr_0' in data: ct_slice = data['arr_0']
                else:
                    print(f"  - Warning: Could not find a valid data key. Skipping.")
                    continue

            if ct_slice.ndim != 2:
                print(f"  - Warning: Expected a 2D array but got shape {ct_slice.shape}. Skipping.")
                continue
        except Exception as e:
            print(f"  - Warning: Could not load file. Error: {e}. Skipping.")
            continue

        # 4. Preprocess the slice
        slice_resized = tf.image.resize(ct_slice[..., tf.newaxis], target_size)
        slice_normalized = slice_resized / 255.0
        batch_to_predict = tf.expand_dims(slice_normalized, axis=0)

        # 5. Get Prediction
        predicted_masks = model.predict(batch_to_predict, verbose=0)
        predicted_mask = predicted_masks[0]

        # 6. Create and Save the visualization for this slice
        plt.figure(figsize=(8, 4))

        # Get patient and slice identifiers for the title and filename
        slice_filename = os.path.splitext(os.path.basename(file_path))[0]
        patient_folder_name = os.path.basename(os.path.dirname(file_path))

        plt.subplot(1, 2, 1)
        plt.title(f"Input: {patient_folder_name} / {slice_filename}")
        plt.imshow(slice_normalized.numpy().squeeze(), cmap='gray')
        plt.axis('off')

        plt.subplot(1, 2, 2)
        plt.title("Predicted Mask")
        plt.imshow(predicted_mask.squeeze(), cmap='gray')
        plt.axis('off')

        # *** MODIFIED NAMING CONVENTION IS HERE ***
        # Save the figure with a unique name: PatientFolder_SliceFilename_prediction.png
        output_filename = f"{patient_folder_name}_{slice_filename}_prediction.png"
        vis_path = os.path.join(results_dir, output_filename)
        plt.tight_layout()
        plt.savefig(vis_path, dpi=100)
        plt.close() # Close the figure to free up memory

    print(f"\n🖼️  All {total_files} prediction images have been saved to: {results_dir}")
    print("\n✅ Prediction completed successfully!")

# Run the main workflow
main(MODEL_PATH, DATA_DIR, RESULTS_DIR)

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Updated Dataset class for multiple numpy files in a folder
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True, target_shape=(1, 256, 256)):
        print(f"Loading numpy files from folder: {folder_path}")
        self.target_shape = target_shape
        npy_files = list(glob.iglob(os.path.join(folder_path, "", "*.npy"), recursive=True))

        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found recursively in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")
        all_images_list = []
        skipped_files = []

        for npy_file in tqdm(npy_files, desc="Loading and processing .npy files"):
            try:
                images = np.load(npy_file)
                original_shape = images.shape
                if len(images.shape) == 2:
                    images = images[None, None, :, :]
                elif len(images.shape) == 3:
                    if images.shape[-1] == 1:
                        images = images[None, :, :, :]
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[-1] == 3:
                        images = images[None, :, :, :]
                        images = np.mean(images, axis=-1, keepdims=True)
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[0] > 1:
                        images = images[:, None, :, :]
                    else:
                        print(f"Skipping {npy_file}: Unexpected 3D shape {original_shape}")
                        skipped_files.append(npy_file)
                        continue
                elif len(images.shape) == 4:
                    if images.shape[1] > 1:
                        if images.shape[1] == 3:
                            images = np.mean(images, axis=1, keepdims=True)
                        elif images.shape[-1] == 3:
                            images = np.mean(images, axis=-1, keepdims=True)
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[-1] == 1:
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[1] == 1:
                            pass
                        else:
                            print(f"Skipping {npy_file}: Unexpected 4D shape with more than 1 channel {images.shape}")
                            skipped_files.append(npy_file)
                            continue
                    elif images.shape[1] == 1:
                        pass
                else:
                    print(f"Skipping {npy_file}: Unexpected shape {original_shape}")
                    skipped_files.append(npy_file)
                    continue

                if len(images.shape) == 4 and images.shape[1:] == self.target_shape:
                    if images.max() > 1.0:
                        images = images.astype(np.float32) / 255.0
                    else:
                        images = images.astype(np.float32)
                    all_images_list.append(images)
                else:
                    print(f"Skipping {npy_file}: Final shape {images.shape} does not match target batch shape (N, {self.target_shape})")
                    skipped_files.append(npy_file)

            except Exception as e:
                print(f"Error loading or processing {npy_file}: {e}")
                skipped_files.append(npy_file)
                continue

        if len(all_images_list) == 0:
            raise ValueError(f"No valid images loaded from {folder_path}. {len(skipped_files)} files were skipped.")

        try:
            self.images = np.concatenate(all_images_list, axis=0)
        except ValueError as e:
            print(f"Error concatenating images: {e}")
            shapes = [img.shape for img in all_images_list]
            print(f"Encountered images with inconsistent shapes during final concat: {shapes}")
            raise ValueError("Images still have inconsistent shapes and cannot be concatenated after filtering.") from e

        print(f"Combined dataset (preloaded): {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()
        image = torch.from_numpy(image).float()

        if self.augment and np.random.random() > 0.5:
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        return image * 2.0 - 1.0

# --- Main Script ---
# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Dataset and DataLoader setup
folder_path = "/content/drive/MyDrive/dataset-1/all_images_only/all_images_only"
train_dataset = MultipleNumpyDataset(folder_path, augment=True)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

# Model configuration
model = UNet2DModel(
    sample_size=256,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)

# Scheduler, Optimizer, Scaler, and Checkpointing
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler()
best_loss = float('inf')
train_losses = [] # Initialize train_losses list

# --- Checkpoint Resuming Section ---
# Define a folder in Google Drive to store checkpoints
checkpoint_path = "/content/drive/MyDrive/dataset-1/ddpm_ct_best_model(1).pt"


start_epoch = 0
global_step = 0

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_loss = checkpoint['loss']
        global_step = checkpoint['global_step']
        print(f"✅ Resuming training from epoch {start_epoch}, with best loss {best_loss:.4f}")
    except KeyError:
        # Fallback for old checkpoints that only save the state_dict
        print("⚠ Checkpoint file is not a full dictionary. Attempting to load model state_dict directly.")
        model.load_state_dict(torch.load(checkpoint_path, map_location=device))
        print("✅ Model weights loaded, but training will start from scratch (epoch 0).")
        start_epoch = 0
        global_step = 0
    except Exception as e:
        print(f"❌ Failed to load checkpoint: {e}")
        print("Starting training from scratch.")
        start_epoch = 0
        global_step = 0
else:
    print("No checkpoint found. Starting training from scratch.")
    start_epoch = 0

# Training parameters
num_epochs = start_epoch + 3
accumulation_steps = 4
save_every = 2
log_every = 20

# --- Training Loop ---
print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        with autocast():
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            clear_memory()

        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1
        train_losses.append(batch_loss) # Append to the losses list

        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })
        global_step += 1

        if batch_idx % log_every == 0:
            pass

    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    # Save best model
    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss, 'global_step': global_step
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, "ddpm_ct_best_model.pt"))
        print(f"✅ New best model saved to Drive! Loss: {best_loss:.4f}")

    # Save periodic checkpoint
    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss, 'global_step': global_step
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt"))
        print(f"📁 Checkpoint saved to Drive for epoch {epoch+1}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

# Save final model state_dict
torch.save(model.state_dict(), os.path.join(checkpoint_dir, "ddpm_ct_final_model.pt"))
print("🎉 Training completed! Final model saved to Drive.")

# Plot training loss
if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

# Test the model with sample generation
print("🖼 Generating test sample...")
model.eval()
with torch.no_grad():
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)
    timesteps = noise_scheduler.timesteps
    for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample
        if i % 100 == 0:
            clear_memory()

    sample = (sample + 1) / 2
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]

    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved to Drive: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")
print("\n🏁 All done!")

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# --- 1. Utility Functions ---
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- 2. Dataset Class ---
class MultipleNumpyDataset(Dataset):
    """Handles loading and preprocessing of multiple .npy files."""
    def __init__(self, folder_path, augment=True, target_shape=(1, 256, 256)):
        print(f"Loading numpy files from folder: {folder_path}")
        self.target_shape = target_shape
        npy_files = list(glob.iglob(os.path.join(folder_path, "**", "*.npy"), recursive=True))

        if not npy_files:
            raise ValueError(f"No .npy files found recursively in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")
        all_images_list = []

        for npy_file in tqdm(npy_files, desc="Processing files"):
            try:
                images = np.load(npy_file)
                if len(images.shape) == 2:
                    images = images[None, None, :, :]
                elif len(images.shape) == 3:
                    if images.shape[0] > 1:
                        images = images[:, None, :, :]
                    else:
                        images = np.transpose(images[None, :, :, :], (0, 3, 1, 2))
                elif len(images.shape) == 4 and images.shape[1] > 1:
                    if images.shape[1] == 3:
                        images = np.mean(images, axis=1, keepdims=True)
                    else:
                        continue
                elif len(images.shape) == 4 and images.shape[-1] > 1:
                    images = np.mean(images, axis=-1, keepdims=True)
                    images = np.transpose(images, (0, 3, 1, 2))

                if images.max() > 1.0:
                    images = images.astype(np.float32) / 255.0
                else:
                    images = images.astype(np.float32)

                if images.shape[1:] == self.target_shape:
                    all_images_list.append(images)
            except Exception as e:
                print(f"Error processing {npy_file}: {e}")

        if not all_images_list:
            raise ValueError("No valid images loaded after processing.")

        self.images = np.concatenate(all_images_list, axis=0)
        print(f"Final combined dataset size: {len(self.images)} images")

        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()
        image = torch.from_numpy(image).float()

        if self.augment and np.random.random() > 0.5:
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        return image * 2.0 - 1.0

# --- 3. Main Script ---
# Dataset and DataLoader setup
folder_path = "/content/drive/MyDrive/dataset-1/all_images_only/all_images_only"
train_dataset = MultipleNumpyDataset(folder_path, augment=True)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

# Model configuration
model = UNet2DModel(
    sample_size=256,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)

# Scheduler, Optimizer, Scaler, and Checkpointing
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler()
best_loss = float('inf')
train_losses = []

# Checkpoint Resuming Section
checkpoint_dir = "/content/checkpoints" # Local Colab path
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, "/content/ddpm_ct_best_model (1) (1).pt")

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['loss']
    global_step = checkpoint['global_step']
    print(f"Resuming training from epoch {start_epoch}")
else:
    print("No checkpoint found. Starting training from scratch.")
    start_epoch = 0
    global_step = 0

# Training parameters
num_epochs = start_epoch + 3
accumulation_steps = 4
save_every = 2
log_every = 20

# --- 4. Training Loop ---
print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        with autocast():
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            clear_memory()

        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1

        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })
        global_step += 1

        if batch_idx % log_every == 0:
            pass

    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss, 'global_step': global_step
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, "ddpm_ct_best_model.pt"))
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss, 'global_step': global_step
        }
        torch.save(checkpoint, os.path.join(checkpoint_dir, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt"))
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

torch.save(model.state_dict(), os.path.join(checkpoint_dir, "ddpm_ct_final_model.pt"))
print("🎉 Training completed!")

# --- 5. Inference Section ---
print("🖼️ Generating test sample...")
model.eval()
ddim_scheduler = DDIMScheduler.from_config(model.config)
ddim_scheduler.set_timesteps(num_inference_steps=50)

with torch.no_grad():
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)

    for i, t in enumerate(tqdm(ddim_scheduler.timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = ddim_scheduler.step(noise_pred, t, sample).prev_sample
        if i % 10 == 0: clear_memory()

    sample = (sample + 1) / 2
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]

    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print(f"• Models saved to Colab: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")
print("\n🏁 All done!")

In [ ]:
import torch
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import numpy as np
import gc
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import os
import glob

# Memory management for Colab
def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Updated Dataset class for multiple numpy files in a folder
class MultipleNumpyDataset(Dataset):
    def __init__(self, folder_path, augment=True, target_shape=(1, 256, 256)):
        print(f"Loading numpy files from folder: {folder_path}")
        self.target_shape = target_shape
        npy_files = list(glob.iglob(os.path.join(folder_path, "**", "*.npy"), recursive=True))

        if len(npy_files) == 0:
            raise ValueError(f"No .npy files found recursively in {folder_path}")

        print(f"Found {len(npy_files)} .npy files")
        all_images_list = []
        skipped_files = []

        for npy_file in tqdm(npy_files, desc="Loading and processing .npy files"):
            try:
                images = np.load(npy_file)
                original_shape = images.shape
                if len(images.shape) == 2:
                    images = images[None, None, :, :]
                elif len(images.shape) == 3:
                    if images.shape[-1] == 1:
                        images = images[None, :, :, :]
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[-1] == 3:
                        images = images[None, :, :, :]
                        images = np.mean(images, axis=-1, keepdims=True)
                        images = np.transpose(images, (0, 3, 1, 2))
                    elif images.shape[0] > 1:
                        images = images[:, None, :, :]
                    else:
                        print(f"Skipping {npy_file}: Unexpected 3D shape {original_shape}")
                        skipped_files.append(npy_file)
                        continue
                elif len(images.shape) == 4:
                    if images.shape[1] > 1:
                        if images.shape[1] == 3:
                            images = np.mean(images, axis=1, keepdims=True)
                        elif images.shape[-1] == 3:
                            images = np.mean(images, axis=-1, keepdims=True)
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[-1] == 1:
                            images = np.transpose(images, (0, 3, 1, 2))
                        elif images.shape[1] == 1:
                            pass
                        else:
                            print(f"Skipping {npy_file}: Unexpected 4D shape with more than 1 channel {images.shape}")
                            skipped_files.append(npy_file)
                            continue
                    elif images.shape[1] == 1:
                        pass
                else:
                    print(f"Skipping {npy_file}: Unexpected shape {original_shape}")
                    skipped_files.append(npy_file)
                    continue

                if len(images.shape) == 4 and images.shape[1:] == self.target_shape:
                    if images.max() > 1.0:
                        images = images.astype(np.float32) / 255.0
                    else:
                        images = images.astype(np.float32)
                    all_images_list.append(images)
                else:
                    print(f"Skipping {npy_file}: Final shape {images.shape} does not match target batch shape (N, {self.target_shape})")
                    skipped_files.append(npy_file)

            except Exception as e:
                print(f"Error loading or processing {npy_file}: {e}")
                skipped_files.append(npy_file)
                continue

        if len(all_images_list) == 0:
            raise ValueError(f"No valid images loaded from {folder_path}. {len(skipped_files)} files were skipped.")

        try:
            self.images = np.concatenate(all_images_list, axis=0)
        except ValueError as e:
            print(f"Error concatenating images: {e}")
            shapes = [img.shape for img in all_images_list]
            print(f"Encountered images with inconsistent shapes during final concat: {shapes}")
            raise ValueError("Images still have inconsistent shapes and cannot be concatenated after filtering.") from e

        print(f"Combined dataset (preloaded): {self.images.shape[0]} images with shape: {self.images.shape}")
        print(f"Data type: {self.images.dtype}, Min: {self.images.min():.3f}, Max: {self.images.max():.3f}")
        self.augment = augment

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = self.images[idx].copy()
        image = torch.from_numpy(image).float()

        if self.augment and np.random.random() > 0.5:
            if np.random.random() > 0.5:
                image = torch.flip(image, [2])

        return image * 2.0 - 1.0

# --- Main Script ---
# Set the device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Dataset and DataLoader setup
folder_path = '/content/drive/MyDrive/dataset-1/all_images_only/all_images_only'
train_dataset = MultipleNumpyDataset(folder_path, augment=True)
train_dataloader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    drop_last=True,
    persistent_workers=True
)

# Model configuration
model = UNet2DModel(
    sample_size=256,
    in_channels=1,
    out_channels=1,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 256),
    down_block_types=("DownBlock2D", "AttnDownBlock2D", "DownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D"),
    attention_head_dim=8,
)
model.to(device)

# Scheduler, Optimizer, Scaler, and Checkpointing
noise_scheduler = DDPMScheduler(num_train_timesteps=1000)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
scaler = GradScaler()
best_loss = float('inf')

# --- Checkpoint Resuming Section ---
checkpoint_path = "/content/ddpm_ct_best_model (1) (1).pt"
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}...")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_loss = checkpoint['loss']
    print(f"Resuming training from epoch {start_epoch}")
else:
    print("No checkpoint found. Starting training from scratch.")
    start_epoch = 0

# Training parameters
num_epochs = start_epoch + 3
accumulation_steps = 4
save_every = 2
log_every = 20
global_step = checkpoint['global_step'] if os.path.exists(checkpoint_path) and 'global_step' in checkpoint else 0

# --- Training Loop ---
print("Starting training...")
print(f"Effective batch size: {train_dataloader.batch_size * accumulation_steps}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    progress_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader), desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch_idx, batch in progress_bar:
        clean_images = batch.to(device, non_blocking=True)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (clean_images.shape[0],), device=device).long()
        noise = torch.randn_like(clean_images)
        noisy_images = noise_scheduler.add_noise(clean_images, noise, timesteps)

        with autocast():
            noise_pred = model(noisy_images, timesteps, return_dict=False)[0]
            loss = F.mse_loss(noise_pred, noise)
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()

        if (batch_idx + 1) % accumulation_steps == 0 or batch_idx == len(train_dataloader) - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            clear_memory()

        batch_loss = loss.item() * accumulation_steps
        epoch_loss += batch_loss
        num_batches += 1

        current_lr = optimizer.param_groups[0]['lr']
        progress_bar.set_postfix({
            'loss': f"{batch_loss:.4f}",
            'avg_loss': f"{epoch_loss / num_batches:.4f}",
            'lr': f"{current_lr:.6f}",
            'step': global_step
        })
        global_step += 1

        if batch_idx % log_every == 0:
            pass

    avg_epoch_loss = epoch_loss / num_batches
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed.")
    print(f"Average loss: {avg_epoch_loss:.4f}")

    if avg_epoch_loss < best_loss:
        best_loss = avg_epoch_loss
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': best_loss, 'global_step': global_step
        }
        torch.save(checkpoint, "ddpm_ct_best_model.pt")
        print(f"✅ New best model saved! Loss: {best_loss:.4f}")

    if (epoch + 1) % save_every == 0:
        checkpoint = {
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(), 'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_epoch_loss, 'global_step': global_step
        }
        torch.save(checkpoint, f"ddpm_ct_checkpoint_epoch_{epoch+1}.pt")
        print(f"📁 Checkpoint saved for epoch {epoch+1}")

    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        reserved = torch.cuda.memory_reserved() / 1024**3
        print(f"GPU Memory: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

    clear_memory()

torch.save(model.state_dict(), "ddpm_ct_final_model.pt")
print("🎉 Training completed!")

# Plot training loss
if train_losses:
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses)
    plt.title('Training Loss')
    plt.xlabel('Batch')
    plt.ylabel('Loss')
    plt.yscale('log')
    plt.grid(True)
    plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')
    plt.show()

# Test the model with sample generation
print("🖼️  Generating test sample...")
model.eval()
with torch.no_grad():
    sample_shape = (1, 1, 256, 256)
    sample = torch.randn(sample_shape, device=device)
    timesteps = noise_scheduler.timesteps
    for i, t in enumerate(tqdm(timesteps, desc="Denoising")):
        with autocast():
            noise_pred = model(sample, t.unsqueeze(0).to(device), return_dict=False)[0]
        sample = noise_scheduler.step(noise_pred, t, sample).prev_sample
        if i % 100 == 0:
            clear_memory()

    sample = (sample + 1) / 2
    sample = torch.clamp(sample, 0, 1)
    sample_np = sample.cpu().numpy()[0, 0]

    plt.figure(figsize=(8, 8))
    plt.imshow(sample_np, cmap='gray')
    plt.title('Generated CT Scan Sample')
    plt.axis('off')
    plt.savefig('generated_ct_sample.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("✅ Sample saved as 'generated_ct_sample.png'")

print("\n📋 Training Summary:")
print(f"• Dataset size: {len(train_dataset):,} images")
print(f"• Image resolution: 256×256")
print(f"• Total epochs: {num_epochs}")
print(f"• Best loss: {best_loss:.4f}")
print(f"• Total training steps: {global_step:,}")
print("• Models saved: ddpm_ct_best_model.pt, ddpm_ct_final_model.pt")
print("\n🏁 All done!")

In [ ]:
import torch
print(torch.__version__)